In [ ]:
#Spark connection
import os
import socket
from pyspark.sql import SparkSession
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = "/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar"
MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        getOrCreate()

In [ ]:
from pyspark.sql import Row

import datetime
users = [
    {
        "id": 1,
        "first_name": "Corrie",
        "last_name": "Van den Oord",
        "email": "cvandenoord0@etsy.com",
        "gender": "male",
        "current_city": "Dallas",
        "phone_numbers": Row(mobile="+1 234 567 8901", home="+1 234 567 8911"),
        "courses": [1, 2],
        "is_customer": True,
        "amount_paid": 1000.55,
        "customer_from": datetime.date(2021, 1, 15),
        "last_updated_ts": datetime.datetime(2021, 2, 10, 1, 15, 0)
    },
    {
        "id": 2,
        "first_name": "Nikolaus",
        "last_name": "Brewitt",
        "email": "nbrewitt1@dailymail.co.uk",
        "gender": "male",
        "current_city": "Houston",
        "phone_numbers":  Row(mobile="+1 234 567 8923", home="1 234 567 8934"),
        "courses": [3],
        "is_customer": True,
        "amount_paid": 900.0,
        "customer_from": datetime.date(2021, 2, 14),
        "last_updated_ts": datetime.datetime(2021, 2, 18, 3, 33, 0)
    },
    {
        "id": 3,
        "first_name": "Orelie",
        "last_name": "Penney",
        "email": "openney2@vistaprint.com",
        "gender": "female",
        "current_city": "",
        "phone_numbers": Row(mobile="+1 714 512 9752", home="+1 714 512 6601"),
        "courses": [2, 4],
        "is_customer": True,
        "amount_paid": 850.55,
        "customer_from": datetime.date(2021, 1, 21),
        "last_updated_ts": datetime.datetime(2021, 3, 15, 15, 16, 55)
    },
    {
        "id": 4,
        "first_name": "Ashby",
        "last_name": "Maddocks",
        "email": "amaddocks3@home.pl",
        "gender": "male",
        "current_city": "San Fransisco",
        "phone_numbers": Row(mobile=None, home=None),
        "courses": [],
        "is_customer": False,
        "amount_paid": None,
        "customer_from": None,
        "last_updated_ts": datetime.datetime(2021, 4, 10, 17, 45, 30)
    },
    {
        "id": 5,
        "first_name": "Kurt",
        "last_name": "Rome",
        "email": "krome4@shutterfly.com",
        "gender": "female",
        "current_city": None,
        "phone_numbers": Row(mobile="+1 817 934 7142", home=None),
        "courses": [],
        "is_customer": False,
        "amount_paid": None,
        "customer_from": None,
        "last_updated_ts": datetime.datetime(2021, 4, 2, 0, 55, 18)
    }
]

In [ ]:
import pandas as pd

df_users = spark.createDataFrame(pd.DataFrame(users))

df_users.show()

In [ ]:
from pyspark.sql.functions import col

df_users.filter(col('id') == 1).show()

In [ ]:
#Пример использования where для фильтрации DF
df_users.where(col('id') == 1).show()

In [ ]:
#Также можно использовать filter. Where и filter аналоги и взаимозаменяемы.
df_users.filter(df_users['id'] == 1).show()

In [ ]:
#Имя колонки и условие можно задать и так
df_users.where('id = 1').show()

In [ ]:
#Не забываем про возможность использовать SQL с помощью createOrReplaceTempView и далее вызовом spark.sql
df_users.createOrReplaceTempView('users')

spark.sql("""
    SELECT *
    FROM users
    WHERE id = 1
"""). \
    show()

In [ ]:
#Дополнительный пример с where, filter и spark.sql
df_users.where(col('is_customer') == True).show(1)

df_users.filter('is_customer = "true"').show(1)

spark.sql('''
    SELECT * FROM users
    WHERE is_customer = "true"
'''). \
    show(1)

In [ ]:
#Знакомимся с функцией isnan
from pyspark.sql.functions import isnan

df_users.select('amount_paid', isnan('amount_paid')).show()

help(isnan)

In [ ]:
df_users.filter('isnan(amount_paid) = True').show()

In [ ]:
#Пример применения условия or и isNull
df_users. \
    select('id', 'current_city'). \
    filter((col('current_city') != 'Dallas') | (col('current_city').isNull())). \
    show()

In [ ]:
#Пример альтернативного синтаксиса без col
df_users. \
    select('id', 'current_city'). \
    filter("current_city != 'Dallas' OR current_city IS NULL"). \
    show()

In [ ]:
#Пример применения between
df_users. \
    select('id', 'email', 'last_updated_ts'). \
    filter(col('last_updated_ts').between('2021-02-15 00:00:00', '2021-03-15 23:59:59')). \
    show()

In [ ]:
df_users. \
    select('id', 'email', 'last_updated_ts'). \
    filter("last_updated_ts BETWEEN '2021-02-15 00:00:00' AND '2021-03-15 23:59:59'"). \
    show()

In [ ]:
df_users. \
    select('id', 'amount_paid'). \
    filter(col('amount_paid').between(850, 900)). \
    show()

In [ ]:
df_users. \
    select('id', 'amount_paid'). \
    filter('amount_paid BETWEEN "850" AND "900"'). \
    show()

In [ ]:
#Примеры использования isNotNull и isNull
df_users. \
    select('id', 'current_city'). \
    filter(col('current_city').isNotNull()). \
    show()

In [ ]:
df_users. \
    select('id', 'current_city'). \
    filter(col('current_city').isNull()). \
    show()


df_users. \
    select('id', 'current_city'). \
    filter('current_city IS NULL'). \
    show()

In [ ]:
df_users. \
    select('id', 'current_city'). \
    filter((col('current_city') == '') | (col('current_city').isNull())). \
    show()


df_users. \
    select('id', 'current_city'). \
    filter("current_city = '' OR current_city IS NULL"). \
    show()

In [ ]:
#Далее пример замены or на isin
df_users. \
    select('id', 'current_city'). \
    filter((col('current_city') == 'Houston') | (col('current_city') == 'Dallas')). \
    show()


df_users. \
    select('id', 'current_city'). \
    filter("current_city = 'Houston' OR current_city = 'Dallas'"). \
    show()

In [ ]:
df_users. \
    select('id', 'current_city'). \
    filter(col('current_city').isin('Houston', 'Dallas')). \
    show()

df_users. \
    select('id', 'current_city'). \
    filter("current_city IN ('Houston', 'Dallas')"). \
    show()

In [ ]:
df_users. \
    filter((col('current_city') == '') | (col('is_customer') == False)). \
    select('id', 'email', 'current_city', 'is_customer'). \
    show()

df_users. \
    filter("current_city = '' OR is_customer = false"). \
    select('id', 'email', 'current_city', 'is_customer'). \
    show()

In [ ]:
spark.stop()